In [ ]:
import json
from dataclasses import dataclass, field
from typing import Optional
import os

from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel, Field
import numpy as np

In [ ]:
# from groq import RateLimitError
# import time


# def call_with_retry(call_function, max_retries=3):

#     for attempt in range(max_retries):

#         try:
#             return call_function()

#         except RateLimitError as e:

#             if attempt == max_retries - 1:
#                 raise

#             print(
#                 "Rate limit reached. "
#                 "Waiting before retry..."
#             )

#             time.sleep(60)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

In [ ]:
load_dotenv("../.env")

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# from app.services.question_generator import generate_question

In [ ]:
with open(
    "../data/candidate_profile.json",
    "r",
    encoding="utf-8"
) as f:
    candidate_profile = json.load(f)

print(candidate_profile)

In [ ]:
class GeneratedQuestion(BaseModel):
    question: str
    topic: str
    difficulty: str
    question_type: str
    expected_concepts: list[str] = Field(
        default_factory=list
    )

class AnswerEvaluation(BaseModel):
    overall_score: float
    technical_accuracy: float
    depth: float
    reasoning: float
    clarity: float
    communication: float
    confidence: float
    strengths: list[str] = Field(default_factory=list)
    weaknesses: list[str] = Field(default_factory=list)
    should_challenge: bool = False
    suggested_follow_up: str = ""
    missing_concepts: list[str] = Field(default_factory=list)

In [ ]:
with open(
    "../data/candidate_profile.json",
    "r",
    encoding="utf-8"
) as f:
    candidate = json.load(f)

with open(
    "../data/interview_blueprint.json",
    "r",
    encoding="utf-8"
) as f:
    blueprint = json.load(f)

print("Target role:", blueprint["target_role"])

In [ ]:
@dataclass
class InterviewState:

    target_role: str

    questions_asked: list[str] = field(
        default_factory=list
    )

    answers: list[str] = field(
        default_factory=list
    )

    evaluations: list[dict] = field(
        default_factory=list
    )
    
    conversation_history: list[dict] = field(
    default_factory=list
)

    topics_covered: list[str] = field(
        default_factory=list
    )

    current_topic: Optional[str] = None

    current_difficulty: str = "medium"

    current_question: Optional[dict] = None

    time_remaining: int = 900

    max_questions: int = 12

    interview_status: str = "not_started"

    follow_up_count: int = 0

    question_count: int = 0

In [ ]:
state = InterviewState(
    target_role=blueprint["target_role"]
)

print(state)

In [ ]:
# state.conversation_history.append({
#     "role": "interviewer",
#     "content": interviewer_response.question
# })

# state.conversation_history.append({
#     "role": "candidate",
#     "content": candidate_answer
# })

In [ ]:
def determine_answer_state(evaluation):

    if evaluation["technical_accuracy"] < 5:
        return "technical_gap"

    if evaluation["depth"] < 5:
        return "shallow"

    if evaluation["reasoning"] < 5:
        return "weak_reasoning"

    if evaluation["overall_score"] >= 8:
        return "strong"

    return "acceptable"

In [ ]:
DIFFICULTY_LEVELS = [
    "easy",
    "medium",
    "hard"
]
def adjust_difficulty(
    current_difficulty,
    answer_state
):

    current_index = DIFFICULTY_LEVELS.index(
        current_difficulty
    )

    if answer_state == "strong":

        new_index = min(
            current_index + 1,
            len(DIFFICULTY_LEVELS) - 1
        )

    elif answer_state in [
        "technical_gap",
        "shallow"
    ]:

        new_index = max(
            current_index - 1,
            0
        )

    else:

        new_index = current_index

    return DIFFICULTY_LEVELS[new_index]



In [ ]:
def choose_next_topic(
    state,
    blueprint
):

    priority_topics = blueprint[
        "priority_topics"
    ]

    for topic in priority_topics:

        if topic not in state.topics_covered:
            return topic

    # fallback
    for topic in priority_topics:
        return topic

    return "General"

In [ ]:
topic = choose_next_topic(
    state,
    blueprint
)

print(topic)

In [ ]:
def should_follow_up(evaluation):

    if evaluation["should_challenge"]:
        return True

    if evaluation["state"] in [
        "shallow",
        "weak_reasoning"
    ]:
        return True

    return False

In [ ]:
MAX_FOLLOW_UPS = 2

def can_follow_up(state):

    return (
        state.follow_up_count
        < MAX_FOLLOW_UPS
    )

In [ ]:
def should_end_interview(state):

    if state.question_count >= state.max_questions:
        return True

    if state.time_remaining <= 0:
        return True

    return False

In [ ]:
def manager_decision(
    state,
    blueprint,
    evaluation=None,
    skeptic_result=None
):
    
    if should_end_interview(state):

        state.interview_status = "completed"

        return {
            "action": "finish",
            "reason": "Interview limit reached"
        }
    
    # Interview hasn't started
    if state.interview_status == "not_started":

        state.interview_status = "in_progress"

        topic = choose_next_topic(
            state,
            blueprint
        )

        return {
            "action": "ask_question",
            "topic": topic,
            "difficulty": state.current_difficulty,
            "reason": "Start interview"
        }

    # We have an evaluation
    if evaluation is not None:

        answer_state = determine_answer_state(
            evaluation
        )

        # Challenge / follow-up
        if (
            should_follow_up(evaluation)
            and can_follow_up(state)
        ):

            state.follow_up_count += 1

            return {
                "action": "follow_up",
                "topic": state.current_topic,
                "difficulty": state.current_difficulty,
                "reason": answer_state
            }

        # Reset follow-up counter
        state.follow_up_count = 0

        # Adapt difficulty
        state.current_difficulty = (
            adjust_difficulty(
                state.current_difficulty,
                answer_state
            )
        )

        # Choose new topic
        topic = choose_next_topic(
            state,
            blueprint
        )

        return {
            "action": "ask_question",
            "topic": topic,
            "difficulty": state.current_difficulty,
            "reason": answer_state
        }

    return {
        "action": "ask_question",
        "topic": choose_next_topic(
            state,
            blueprint
        ),
        "difficulty": state.current_difficulty,
        "reason": "Default"
    }

In [ ]:
decision = manager_decision(
    state,
    blueprint=blueprint
)

print(decision)

In [ ]:
strong_evaluation = {
    "overall_score": 9,
    "technical_accuracy": 9,
    "depth": 9,
    "reasoning": 8,
    "should_challenge": False,
    "state": "strong"
}

decision = manager_decision(
    state,
    evaluation=strong_evaluation,
    blueprint=blueprint
)

print(decision)

In [ ]:
weak_evaluation = {
    "overall_score": 4,
    "technical_accuracy": 4,
    "depth": 3,
    "reasoning": 4,
    "should_challenge": False,
    "state": "technical_gap"
}

decision = manager_decision(
    state,
    evaluation=weak_evaluation,
    blueprint=blueprint
)

print(decision)

In [ ]:
shallow_evaluation = {
    "overall_score": 6,
    "technical_accuracy": 7,
    "depth": 4,
    "reasoning": 4,
    "should_challenge": False,
    "state": "shallow"
}

decision = manager_decision(
    state,
    evaluation=shallow_evaluation,
    blueprint=blueprint
)

print(decision)

In [ ]:
suspicious_evaluation = {
    "overall_score": 7,
    "technical_accuracy": 7,
    "depth": 6,
    "reasoning": 6,
    "should_challenge": True,
    "state": "acceptable"
}

decision = manager_decision(
    state,
    evaluation=suspicious_evaluation,
    blueprint=blueprint
)

print(decision)

In [ ]:
def should_end_interview(state):

    if state.question_count >= state.max_questions:
        return True

    if state.time_remaining <= 0:
        return True

    return False

In [ ]:
state.question_count = 12

print(
    should_end_interview(state)
)

In [ ]:
state = InterviewState(
    target_role=blueprint["target_role"]
)

decision = manager_decision(
    state,
    blueprint=blueprint
)
print(decision)

In [ ]:
def generate_question(
    topic: str,
    difficulty: str,
    question_type: str,
    previous_questions: list[str] | None = None
):
    
    if previous_questions is None:
        previous_questions = []

    previous_text = "\n".join(
        f"- {q}"
        for q in previous_questions[-10:]
    )

    prompt = f"""
You are an expert technical interviewer.

Generate ONE interview question.

Target topic:
{topic}

Difficulty:
{difficulty}

Question type:
{question_type}

Previously asked questions:
{previous_text if previous_text else "None"}

Rules:

1. The question must test the specified topic.
2. Match the requested difficulty.
3. Do not repeat or closely rephrase previous questions.
4. The question should be appropriate for a technical interview.
5. Return expected concepts that a strong answer should contain.
6. Return ONLY valid JSON.

Return:

{{
    "question": "...",
    "topic": "...",
    "difficulty": "...",
    "question_type": "...",
    "expected_concepts": [
        "...",
        "..."
    ]
}}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert technical interviewer. "
                    "Return only valid JSON."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.7
    )

    response_text = response.choices[0].message.content

    result = json.loads(response_text)

    return GeneratedQuestion.model_validate(result)

In [ ]:
question = generate_question(
    topic=decision["topic"],
    difficulty=decision["difficulty"],
    question_type="technical",
    previous_questions=state.questions_asked
)

In [ ]:
print(question.question)
print(question.expected_concepts)

In [ ]:
candidate_answer = """
I chose XGBoost because it is a gradient boosting algorithm
that works very well with structured and tabular data. It can
capture nonlinear relationships and interactions between
features. It also provides useful feature importance and
usually performs well without requiring extensive feature
scaling.
"""

In [ ]:
question_for_evaluation = question.model_dump()
print(
    json.dumps(
        question_for_evaluation,
        indent=2
    )
)

In [ ]:
evaluation_schema = AnswerEvaluation.model_json_schema()

print(json.dumps(
    evaluation_schema,
    indent=2
))

In [ ]:
EVALUATOR_PROMPT = """
You are the Answer Evaluation Agent for InterviewHive.

You are an expert technical interviewer evaluating a candidate's
answer to an interview question.

Evaluate ONLY the candidate's answer against:
- the interview question
- the expected concepts
- the target role when relevant

Do not evaluate the candidate as a person.

EVALUATION DIMENSIONS

Score every dimension from 0 to 10.

1. technical_accuracy
   - Are the technical statements correct?
   - Penalize incorrect technical claims strongly.

2. depth
   - How thoroughly does the answer explain the concept?
   - A short but correct answer can still score well.

3. reasoning
   - Does the candidate explain why, how, trade-offs,
     implications, or decision-making?

4. clarity
   - Is the answer understandable and logically organized?

5. communication
   - Does the candidate communicate the answer effectively,
     directly, and professionally?

6. confidence
   - Evaluate how appropriately and decisively the answer
     is communicated.
   - Do not invent confidence evidence.

OVERALL SCORE

overall_score must represent the overall quality of the answer.

Technical accuracy and relevance to the question are especially
important.

SCORING GUIDE

0-2 = very poor
3-4 = weak
5-6 = acceptable
7-8 = strong
9-10 = excellent

STRENGTHS

Only mention things the candidate actually did well.

WEAKNESSES

Mention specific problems in the answer.

MISSING CONCEPTS

Only include concepts from expected_concepts that are genuinely
missing or insufficiently addressed.

Do not invent missing concepts.

CHALLENGE

Set should_challenge to true if:
- the candidate makes an incorrect technical claim,
- the answer is too vague to establish understanding,
- an important concept is misunderstood,
- or a follow-up would meaningfully test the candidate.

Otherwise set it to false.

FOLLOW-UP

If should_challenge is true, provide a useful follow-up question
targeting the biggest weakness.

If no follow-up is needed, return an empty string.

IMPORTANT OUTPUT RULES

Return ONLY valid JSON.

Use EXACTLY these field names:

{
    "overall_score": 0,
    "technical_accuracy": 0,
    "depth": 0,
    "reasoning": 0,
    "clarity": 0,
    "communication": 0,
    "confidence": 0,
    "strengths": [],
    "weaknesses": [],
    "should_challenge": false,
    "suggested_follow_up": "",
    "missing_concepts": []
}

Do NOT use alternative field names such as:
- score
- feedback
- accuracy
- follow_up
- explanation

Every field must be present.

Return ONLY the JSON object.
"""

In [ ]:
def evaluate_answer(
    question,
    candidate_answer
):

    evaluation_input = {
        "target_role": blueprint["target_role"],

        "question": question["question"],

        "topic": question.get("topic", ""),

        "category": question.get("category", ""),

        "difficulty": question.get("difficulty", ""),

        "question_type": question.get("question_type", ""),

        "expected_concepts": question.get(
            "expected_concepts",
            []
        ),

        "candidate_answer": candidate_answer
    }

    prompt = f"""
Evaluate the candidate's interview answer using the provided
question and expected concepts.

QUESTION CONTEXT
{json.dumps(
    evaluation_input,
    indent=2,
    ensure_ascii=False
)}

REQUIRED JSON SCHEMA
{json.dumps(
    evaluation_schema,
    indent=2
)}

Return ONLY valid JSON matching the schema.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",

        messages=[
            {
                "role": "system",
                "content": EVALUATOR_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    response_text = response.choices[0].message.content

    result = json.loads(response_text)

    return AnswerEvaluation.model_validate(result)

In [ ]:
evaluation = evaluate_answer(
    question_for_evaluation,
    candidate_answer
)

print(evaluation.model_dump())

In [ ]:
def determine_answer_state(evaluation):

    if evaluation["technical_accuracy"] < 5:
        return "technical_gap"

    if evaluation["depth"] < 5:
        return "shallow"

    if evaluation["reasoning"] < 5:
        return "weak_reasoning"

    if evaluation["overall_score"] >= 8:
        return "strong"

    return "acceptable"

In [ ]:
# WEIGHTS = {
#     "technical_accuracy": 0.30,
#     "depth": 0.20,
#     "reasoning": 0.20,
#     "clarity": 0.10,
#     "communication": 0.10,
#     "confidence": 0.10
# }
# def calculate_weighted_score(evaluation):

#     score = (
#         evaluation.technical_accuracy
#         * WEIGHTS["technical_accuracy"]

#         + evaluation.depth
#         * WEIGHTS["depth"]

#         + evaluation.reasoning
#         * WEIGHTS["reasoning"]

#         + evaluation.clarity
#         * WEIGHTS["clarity"]

#         + evaluation.communication
#         * WEIGHTS["communication"]

#         + evaluation.confidence
#         * WEIGHTS["confidence"]
#     )

#     return round(score, 2)


In [ ]:
def create_evaluation_signal(evaluation):

    answer_state = determine_answer_state(
        evaluation.model_dump()
    )

    return {
        "overall_score": evaluation.overall_score,
        "technical_accuracy": evaluation.technical_accuracy,
        "depth": evaluation.depth,
        "reasoning": evaluation.reasoning,
        "clarity": evaluation.clarity,
        "communication": evaluation.communication,
        "confidence": evaluation.confidence,
        "state": answer_state,
        "should_challenge": evaluation.should_challenge,
        "suggested_follow_up": evaluation.suggested_follow_up,
        "missing_concepts": evaluation.missing_concepts
    }

In [ ]:
signal = create_evaluation_signal(
    evaluation
)

print(
    json.dumps(
        signal,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
state.answers.append(
    candidate_answer
)

state.evaluations.append(
    signal
)

In [ ]:
print("Answers:", len(state.answers))
print("Evaluations:", len(state.evaluations))

In [ ]:
next_decision = manager_decision(
    state,
    evaluation=signal,
    blueprint=blueprint
)

print(
    json.dumps(
        next_decision,
        indent=2
    )
)

In [ ]:
next_question = generate_question(
    topic=next_decision["topic"],
    difficulty=next_decision["difficulty"],
    question_type="technical",
    previous_questions=state.questions_asked
)
print(next_question.question)

In [ ]:
def check_question_history(
    new_question: str,
    asked_questions: list[str],
    threshold: float = 0.80
):

    if not asked_questions:
        return {
            "is_duplicate": False,
            "similarity": 0.0,
            "matched_question": None
        }

    new_embedding = model.encode(
        [new_question],
        normalize_embeddings=True
    ).astype("float32")

    history_embeddings = model.encode(
        asked_questions,
        normalize_embeddings=True
    ).astype("float32")

    scores = np.matmul(
        history_embeddings,
        new_embedding[0]
    )

    best_index = int(
        np.argmax(scores)
    )

    best_score = float(
        scores[best_index]
    )

    return {
        "is_duplicate": best_score >= threshold,
        "similarity": best_score,
        "matched_question": asked_questions[best_index]
    }

In [ ]:
def is_question_repetitive(
    new_question,
    previous_questions,
    threshold=0.85
):
    
    if not previous_questions:
        return False

    result = check_question_history(
        new_question,
        previous_questions,
        threshold=threshold
    )

    return result

In [ ]:
def generate_unique_question(
    topic,
    difficulty,
    question_type,
    previous_questions,
    max_attempts=3
):

    for attempt in range(max_attempts):

        question = generate_question(
            topic=topic,
            difficulty=difficulty,
            question_type=question_type,
            previous_questions=previous_questions
        )

        is_duplicate = is_question_repetitive(
            question.question,
            previous_questions
        )

        if not is_duplicate:
            return question

        print(
            f"Question too similar. "
            f"Regenerating ({attempt + 1}/{max_attempts})..."
        )

    # Return last generated question if all attempts
    # were considered too similar
    return question

In [ ]:
question = generate_unique_question(
    topic=decision["topic"],
    difficulty=decision["difficulty"],
    question_type="technical",
    previous_questions=state.questions_asked
)

In [ ]:
print(question.question)

In [ ]:
state.current_question = question.model_dump()

state.questions_asked.append(
    question.question
)

state.current_topic = question.topic

if question.topic not in state.topics_covered:
    state.topics_covered.append(question.topic)

state.question_count += 1

In [ ]:
decision = manager_decision(
    state,
    blueprint=blueprint
)

print(decision)

In [ ]:
question = generate_question(
    topic=decision["topic"],
    difficulty=decision["difficulty"],
    question_type="technical",
    previous_questions=state.questions_asked
)

print(question.question)

In [ ]:
candidate_answer = """
I would evaluate the model using appropriate classification
metrics such as precision, recall and F1 score. The choice
depends on whether false positives or false negatives are
more important for the application.
"""

In [ ]:
question_for_evaluation = question.model_dump()

evaluation = evaluate_answer(
    question_for_evaluation,
    candidate_answer
)

signal = create_evaluation_signal(
    evaluation
)

In [ ]:
state.answers.append(candidate_answer)
state.evaluations.append(signal)

In [ ]:
next_decision = manager_decision(
    state,
    evaluation=signal,
    blueprint=blueprint
)

print(next_decision)

In [ ]:
def run_interview_turn(
    state,
    blueprint,
    candidate_answer=None
):

    #Evaluate previous answer

    evaluation = None
    signal = None

    if candidate_answer is not None:

        current_question = state.current_question

        evaluation = evaluate_answer(
            current_question,
            candidate_answer
        )

        signal = create_evaluation_signal(
            evaluation
        )

        state.answers.append(
            candidate_answer
        )

        state.evaluations.append(
            signal
        )

    #Skeptic Agent

    skeptic_result = None

    if candidate_answer is not None:

        skeptic_result = skeptic_agent(
            candidate_answer=candidate_answer,
            question=state.current_question,
            candidate_profile=candidate_profile
        )

    #Manager decides next action

    decision = manager_decision(
        state,
        evaluation=signal,
        skeptic_result=(
            skeptic_result.model_dump()
            if skeptic_result
            else None
        ),
        blueprint=blueprint
    )

    #Check termination

    if decision["action"] == "finish":

        state.interview_status = "completed"

        return {
            "status": "completed",
            "decision": decision,
            "question": None,
            "evaluation": (
                evaluation.model_dump()
                if evaluation
                else None
            ),
            "skeptic": (
                skeptic_result.model_dump()
                if skeptic_result
                else None
            )
        }

    #Handle challenge

    if decision["action"] == "challenge":

        challenge_question = (
            skeptic_result.challenge_question
        )

        state.current_question = {
            "question": challenge_question,
            "topic": decision["topic"],
            "difficulty": decision["difficulty"],
            "question_type": "challenge",
            "expected_concepts": []
        }

        state.questions_asked.append(
            challenge_question
        )

        state.question_count += 1

        return {
            "status": "in_progress",
            "decision": decision,
            "question": state.current_question,
            "evaluation": (
                evaluation.model_dump()
                if evaluation
                else None
            ),
            "skeptic": (
                skeptic_result.model_dump()
                if skeptic_result
                else None
            )
        }

    #Generate next question

    question = generate_unique_question(
        topic=decision["topic"],
        difficulty=decision["difficulty"],
        question_type="technical",
        previous_questions=state.questions_asked
    )

    #Update interview state

    state.current_question = (
        question.model_dump()
    )

    state.current_topic = (
        question.topic
    )

    state.current_difficulty = (
        question.difficulty
    )

    state.questions_asked.append(
        question.question
    )

    if question.topic not in state.topics_covered:

        state.topics_covered.append(
            question.topic
        )

    state.question_count += 1

    #Return interview turn

    return {
        "status": "in_progress",

        "decision": decision,

        "question": question.model_dump(),

        "evaluation": (
            evaluation.model_dump()
            if evaluation
            else None
        ),

        "skeptic": (
            skeptic_result.model_dump()
            if skeptic_result
            else None
        )
    }

In [ ]:
state = InterviewState(
    target_role=blueprint["target_role"]
)

turn = run_interview_turn(
    state,
    blueprint
)

print(turn["question"]["question"])

In [ ]:
INTERVIEWER_PROMPT = """
You are the interviewer in a realistic technical job interview.

Your job is to communicate naturally with the candidate.

You receive:
- candidate profile
- target role
- manager decision
- generated question
- previous conversation

Rules:

1. Ask exactly one question at a time.
2. Be professional but conversational.
3. Do not give the candidate the answer.
4. Do not evaluate the candidate.
5. Do not reveal internal scores or agent decisions.
6. If the action is a follow-up, connect naturally to the candidate's
   previous answer.
7. If the action is a new question, transition naturally.
8. Questions should sound like something a real interviewer would ask.
9. Reference the candidate's resume when relevant.
10. Keep the response concise.

Return ONLY valid JSON:

{
    "message": "...",
    "question": "..."
}
"""

In [ ]:
class InterviewerResponse(BaseModel):

    message: str
    question: str

In [ ]:
def interviewer_agent(
    decision,
    question,
    candidate_profile,
    conversation_history=None,
    candidate_answer=None
):

    if conversation_history is None:
        conversation_history = []

    input_data = {
        "candidate_profile": candidate_profile,
        "manager_decision": decision,
        "question": question,
        "conversation_history": conversation_history[-6:],
        "candidate_answer": candidate_answer
    }

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": INTERVIEWER_PROMPT
            },
            {
                "role": "user",
                "content": json.dumps(
                    input_data,
                    indent=2,
                    ensure_ascii=False
                )
            }
        ],
        temperature=0.6
    )

    response_text = response.choices[0].message.content

    parsed = json.loads(response_text)

    return InterviewerResponse.model_validate(
        parsed
    )

In [ ]:
interviewer_response = interviewer_agent(
    decision=turn["decision"],
    question=turn["question"],
    candidate_profile=candidate_profile,
    conversation_history=state.answers
)

In [ ]:
print(
    interviewer_response.message
)

print(
    interviewer_response.question
)

## Skeptic agent

In [ ]:
class SkepticResponse(BaseModel):
    should_challenge: bool
    concern: str
    challenge_question: str

In [ ]:
SKEPTIC_PROMPT = """
You are a skeptical technical interviewer.

Your job is to identify claims in a candidate's answer that
may be:

- unsupported
- exaggerated
- vague
- technically questionable
- missing measurable evidence
- inconsistent with the candidate's resume

You do NOT automatically challenge every answer.

Only recommend a challenge when there is a meaningful reason.

Examples of claims worth challenging:

"I improved performance by 80%."
"I achieved 99% accuracy."
"Our system handles millions of users."
"I designed the entire architecture."
"The model was highly accurate."

For measurable claims, look for:
- baseline
- measurement method
- dataset
- evaluation metric
- experimental setup
- candidate's specific contribution

Return ONLY valid JSON:

{
    "should_challenge": true/false,
    "concern": "...",
    "challenge_question": "..."
}
"""

In [ ]:
def skeptic_agent(
    candidate_answer,
    question,
    candidate_profile
):

    input_data = {
        "question": question,
        "candidate_answer": candidate_answer,
        "candidate_profile": candidate_profile
    }

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": SKEPTIC_PROMPT
            },
            {
                "role": "user",
                "content": json.dumps(
                    input_data,
                    indent=2,
                    ensure_ascii=False
                )
            }
        ],
        temperature=0
    )

    response_text = response.choices[0].message.content

    parsed = json.loads(response_text)

    return SkepticResponse.model_validate(
        parsed
    )

In [ ]:
candidate_answer = """
I improved the model accuracy by 80% and made the system
10 times faster.
"""

skeptic_result = skeptic_agent(
    candidate_answer=candidate_answer,
    question=state.current_question,
    candidate_profile=candidate_profile
)

print(
    skeptic_result.model_dump()
)

### testing

In [ ]:
state = InterviewState(
    target_role=blueprint["target_role"]
)
turn = run_interview_turn(
    state,
    blueprint
)

print(turn["question"]["question"])

In [ ]:
candidate_answer = """
I increased the accuracy of the model by 80% and reduced
the processing time by 90%. I was responsible for the
entire machine learning pipeline.
"""

In [ ]:
turn = run_interview_turn(
    state,
    blueprint,
    candidate_answer=candidate_answer
)
print(turn)

In [ ]:
print("EVALUATION")
print(turn["evaluation"])

print("\nDECISION")
print(turn["decision"])

print("\nQUESTION")
print(turn["question"])

In [ ]:
print("\nSKEPTIC")
print(turn["skeptic"])

## Judge agent

In [ ]:
class FinalInterviewReport(BaseModel):
    overall_score: float
    technical_score: float
    problem_solving_score: float
    communication_score: float
    confidence_score: float
    depth_score: float

    strengths: list[str] = Field(
        default_factory=list
    )

    weaknesses: list[str] = Field(
        default_factory=list
    )

    red_flags: list[str] = Field(
        default_factory=list
    )

    recommended_topics: list[str] = Field(
        default_factory=list
    )

    summary: str

In [ ]:
JUDGE_PROMPT = """
You are the final judge of a technical job interview.

You must evaluate the candidate based on the COMPLETE interview.

You receive:

- candidate profile
- target role
- questions
- answers
- answer-level evaluations
- topics covered
- interview performance

Do not judge the candidate based on a single answer.

Consider:

1. Technical knowledge
2. Problem solving
3. Communication
4. Confidence
5. Technical depth
6. Consistency
7. Ability to explain projects
8. Ability to defend technical decisions
9. Evidence supporting claims
10. Overall interview performance

Important:

- Do not reward buzzwords without understanding.
- Penalize unsupported or exaggerated claims.
- Consider the difficulty of the questions.
- Consider improvement or decline throughout the interview.
- Do not invent experience that was not demonstrated.

Return ONLY valid JSON.

{
    "overall_score": 0,
    "technical_score": 0,
    "problem_solving_score": 0,
    "communication_score": 0,
    "confidence_score": 0,
    "depth_score": 0,
    "strengths": [],
    "weaknesses": [],
    "red_flags": [],
    "recommended_topics": [],
    "summary": ""
}
"""

In [ ]:
def judge_agent(
    candidate_profile,
    target_role,
    questions,
    answers,
    evaluations,
    topics_covered
):

    interview_data = {
        "candidate_profile": candidate_profile,
        "target_role": target_role,
        "questions": questions,
        "answers": answers,
        "evaluations": evaluations,
        "topics_covered": topics_covered
    }

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": JUDGE_PROMPT
            },
            {
                "role": "user",
                "content": json.dumps(
                    interview_data,
                    indent=2,
                    ensure_ascii=False
                )
            }
        ],
        temperature=0
    )

    response_text = response.choices[0].message.content

    parsed = json.loads(response_text)

    return FinalInterviewReport.model_validate(
        parsed
    )

In [ ]:
state = InterviewState(
    target_role=blueprint["target_role"]
)

In [ ]:
turn = run_interview_turn(
    state=state,
    blueprint=blueprint
)

print(turn["question"]["question"])

In [ ]:
answer_1 = """
I used XGBoost because it performs well on structured
tabular data and can capture nonlinear relationships.
I compared it with logistic regression and random forest
and found that XGBoost performed better on validation data.
"""

In [ ]:
turn = run_interview_turn(
    state=state,
    blueprint=blueprint,
    candidate_answer=answer_1
)

print(turn["question"]["question"])

In [ ]:
answers = [
    """
    I chose XGBoost because it performs well on structured
    data and captures nonlinear relationships.
    """,

    """
    I used cross validation to estimate how well the model
    would generalize to unseen data.
    """,

    """
    I evaluated the classification model using precision,
    recall, F1 score and the confusion matrix.
    """,

    """
    I improved preprocessing by handling missing values,
    removing irrelevant features and standardizing the
    data where appropriate.
    """,

    """
    I used feature importance to understand which features
    contributed most to the model predictions.
    """
]

In [ ]:
for answer in answers:

    turn = run_interview_turn(
        state=state,
        blueprint=blueprint,
        candidate_answer=answer
    )

    print("=" * 60)
    print("QUESTION:")
    print(turn["question"]["question"])

    print("\nDECISION:")
    print(turn["decision"])

In [ ]:
print("Questions:", len(state.questions_asked))
print("Answers:", len(state.answers))
print("Evaluations:", len(state.evaluations))
print("Topics:", state.topics_covered)

In [ ]:
final_report = judge_agent(
    candidate_profile=candidate_profile,
    target_role=blueprint["target_role"],
    questions=state.questions_asked,
    answers=state.answers,
    evaluations=state.evaluations,
    topics_covered=state.topics_covered
)

print(
    json.dumps(
        final_report.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

### helper to display report 

In [ ]:
def display_final_report(report):

    print("=" * 60)
    print("FINAL INTERVIEW REPORT")
    print("=" * 60)

    print(f"\nOverall Score: {report.overall_score}/100")

    print("\nScores")
    print("-" * 30)

    print(
        f"Technical Knowledge : {report.technical_score}/100"
    )

    print(
        f"Problem Solving     : {report.problem_solving_score}/100"
    )

    print(
        f"Communication       : {report.communication_score}/100"
    )

    print(
        f"Confidence          : {report.confidence_score}/100"
    )

    print(
        f"Technical Depth     : {report.depth_score}/100"
    )

    print("\nStrengths")
    for item in report.strengths:
        print(f"✓ {item}")

    print("\nWeaknesses")
    for item in report.weaknesses:
        print(f"⚠ {item}")

    print("\nRed Flags")
    for item in report.red_flags:
        print(f"⚠ {item}")

    print("\nRecommended Preparation")
    for item in report.recommended_topics:
        print(f"→ {item}")

    print("\nSummary")
    print(report.summary)

In [ ]:
display_final_report(final_report)

## full interview pipeline

In [ ]:
def simulated_candidate_answer(question, turn_number):
# temporary
    answers = [
        """
        I chose XGBoost because it performs well on structured
        tabular data and can capture nonlinear relationships.
        I also compared it with simpler models before selecting it.
        """,

        """
        I evaluated the model using cross validation and metrics
        such as precision, recall and F1 score. I selected the
        metric based on the business impact of false positives
        and false negatives.
        """,

        """
        I handled missing values, removed irrelevant features
        and performed feature engineering before training the model.
        """,

        """
        I used feature importance to understand which variables
        contributed most to the predictions. I also checked the
        validation performance to make sure the model was not
        simply overfitting.
        """,

        """
        The main challenge was balancing model performance with
        interpretability. I compared multiple approaches and
        selected the one that gave the best validation results
        while remaining practical for deployment.
        """
    ]

    if turn_number < len(answers):
        return answers[turn_number]

    return """
    I would investigate the issue by checking the data,
    validation results and assumptions before changing the model.
    """

In [ ]:
def run_full_interview(
    candidate_profile,
    blueprint,
    max_questions=5
):

    # Initialize interview

    state = InterviewState(
        target_role=blueprint["target_role"]
    )

    state.interview_status = "in_progress"

    transcript = []

    # First question

    turn = run_interview_turn(
        state=state,
        blueprint=blueprint
    )

    question = turn["question"]

    transcript.append({
        "role": "interviewer",
        "content": question["question"]
    })

    # Interview loop

    while (
        state.question_count < max_questions
        and state.interview_status != "completed"
    ):

        print("\n" + "=" * 70)

        print(
            f"QUESTION {state.question_count}"
        )

        print("=" * 70)

        print(
            question["question"]
        )

        
        # Simulate candidate
        answer = simulated_candidate_answer(
            question["question"],
            state.question_count - 1
        )

        print("\nCANDIDATE:")
        print(answer)

        transcript.append({
            "role": "candidate",
            "content": answer
        })

        
        # Process answer + generate next turn

        turn = run_interview_turn(
            state=state,
            blueprint=blueprint,
            candidate_answer=answer
        )

        
        # Store evaluation if available

        if turn["evaluation"] is not None:

            print("\nEVALUATION:")

            print(
                turn["evaluation"]
            )

        
        # Check completion

        if turn["status"] == "completed":

            break

        question = turn["question"]

        transcript.append({
            "role": "interviewer",
            "content": question["question"]
        })

    # Final Judge

    print("\n" + "=" * 70)
    print("INTERVIEW COMPLETE")
    print("=" * 70)

    final_report = judge_agent(
        candidate_profile=candidate_profile,
        target_role=blueprint["target_role"],
        questions=state.questions_asked,
        answers=state.answers,
        evaluations=state.evaluations,
        topics_covered=state.topics_covered
    )

    # Return everything

    return {
        "state": state,
        "transcript": transcript,
        "final_report": final_report
    }

In [ ]:
full_result = run_full_interview(
    candidate_profile=candidate_profile,
    blueprint=blueprint,
    max_questions=5
)
display_final_report(
    full_result["final_report"]
)

In [ ]:
state = full_result["state"]

print("Questions:", len(state.questions_asked))
print("Answers:", len(state.answers))
print("Evaluations:", len(state.evaluations))
print("Topics:", state.topics_covered)

In [ ]:
for message in full_result["transcript"]:
    print(
        f"{message['role'].upper()}:"
    )
    print(message["content"])
    print()